# ASEAN PTCST-v2 — resume from saved V2 data

This notebook skips V1/V2 dataset construction. It restores a prebuilt V2 dataset and saved forecast runs from Google Drive, then runs training only when explicitly enabled, followed by forecast evaluation, ensemble calibration, risk-aversion selection and C0/C1/C2 portfolio backtests.

2024–2025 remain development evidence, not an untouched final holdout.

In [ ]:
# Cell 1 — mount Drive and clone current code
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
from pathlib import Path
import subprocess, sys, shutil, pandas as pd
REPO = Path('/content/kltn')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','https://github.com/maiphuowng205/kltn.git',str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-colab.txt')], check=True)
DRIVE = Path('/content/drive/MyDrive')
print('Commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())

In [ ]:
# Cell 2 — restore the already-built V2 dataset; do not rebuild it here
V2_DATA = Path('/content/asean_v2')
V2_DRIVE_CANDIDATES = [
    DRIVE/'kltn'/'asean_v2',
    DRIVE/'kltn'/'asean_v2_dataset',
    DRIVE/'kltn'/'asean_v2_development'/'asean_v2',
    DRIVE/'kltn'/'asean_v2_development'/'dataset',
]
def is_v2_root(path):
    return (path/'model_ready'/'weekly_features_targets_v2').exists() and (path/'curated'/'daily_panel_v2').exists()
source_v2 = next((p for p in V2_DRIVE_CANDIDATES if is_v2_root(p)), None)
if source_v2 is None and is_v2_root(V2_DATA): source_v2 = V2_DATA
if source_v2 is None:
    raise FileNotFoundError('A complete prebuilt V2 dataset was not found on Drive. Run Notebook 11 Cell 3 once and save /content/asean_v2 to MyDrive/kltn/asean_v2; this Notebook 12 never rebuilds it.')
if V2_DATA.exists() and source_v2 != V2_DATA: shutil.rmtree(V2_DATA)
if source_v2 != V2_DATA: shutil.copytree(source_v2, V2_DATA)
print('Using V2 dataset:', V2_DATA)

In [ ]:
# Cell 3 — restore saved training outputs, or train only if explicitly requested
RUN_FORECAST_TRAINING = False  # keep False to avoid retraining saved seeds
V2_RUN = Path('/content/asean_v2_runs/pooled_ptcst')
DRIVE_RUN = DRIVE/'kltn'/'asean_v2_development'/'pooled_ptcst'
seeds = [7, 19, 31, 43, 59]
if RUN_FORECAST_TRAINING:
    if V2_RUN.exists(): shutil.rmtree(V2_RUN)
    subprocess.run([sys.executable,str(REPO/'scripts/run_asean_v2_forecasts.py'),'--data-root',str(V2_DATA),'--run-root',str(V2_RUN),'--epochs','100','--seeds','7,19,31,43,59'], check=True)
else:
    if not DRIVE_RUN.exists(): raise FileNotFoundError(f'Missing saved training outputs: {DRIVE_RUN}')
    if V2_RUN.exists(): shutil.rmtree(V2_RUN)
    shutil.copytree(DRIVE_RUN, V2_RUN)
for seed in seeds:
    required = V2_RUN/f'seed_{seed}'/'development_predictions.npz'
    if not required.exists(): raise FileNotFoundError(required)
print('Forecast runs ready:', V2_RUN)

In [ ]:
# Cell 4 — locked forecast metrics for all five seeds
METRICS = Path('/content/asean_v2_forecast_metrics')
if METRICS.exists(): shutil.rmtree(METRICS)
command = [sys.executable,str(REPO/'scripts/evaluate_asean_v2_forecasts.py'),'--output-dir',str(METRICS)]
for seed in seeds:
    command += ['--input', f'PTCST-v2_seed_{seed}={V2_RUN/f"seed_{seed}"/"development_predictions.npz"}']
subprocess.run(command, check=True)
forecast_summary = pd.read_csv(METRICS/'forecast_metrics_summary.csv')
display(forecast_summary)

In [ ]:
# Cell 5 — rank-normalized five-seed ensemble and validation-only calibration
V21_OUT = V2_RUN/'v2_1_ensemble'
if V21_OUT.exists(): shutil.rmtree(V21_OUT)
subprocess.run([sys.executable,str(REPO/'scripts/run_asean_v21_ensemble.py'),'--run-root',str(V2_RUN),'--output-dir',str(V21_OUT),'--seeds','7,19,31,43,59'], check=True)
print('Ensemble ready:', V21_OUT)

In [ ]:
# Cell 6 — evaluate ensemble and save forecast results
ENSEMBLE_METRICS = Path('/content/asean_v21_ensemble_metrics')
if ENSEMBLE_METRICS.exists(): shutil.rmtree(ENSEMBLE_METRICS)
subprocess.run([sys.executable,str(REPO/'scripts/evaluate_asean_v2_forecasts.py'),'--output-dir',str(ENSEMBLE_METRICS),'--input',f'PTCST-v2.1-Ensemble={V21_OUT/"development_ensemble.npz"}'], check=True)
display(pd.read_csv(ENSEMBLE_METRICS/'forecast_metrics_summary.csv'))
DRIVE_OUT = DRIVE/'kltn'/'asean_v2_development'
(DRIVE_OUT/'forecast_evaluation').mkdir(parents=True, exist_ok=True)
shutil.copytree(METRICS, DRIVE_OUT/'forecast_evaluation'/'seed_metrics', dirs_exist_ok=True)
shutil.copytree(ENSEMBLE_METRICS, DRIVE_OUT/'forecast_evaluation'/'ensemble_metrics', dirs_exist_ok=True)
shutil.copytree(V21_OUT, DRIVE_OUT/'v2_1_ensemble', dirs_exist_ok=True)
print('Forecast outputs saved to:', DRIVE_OUT)

In [ ]:
# Cell 7 — select risk aversion on validation only
LAMBDA_GRID = [2, 5, 10, 20, 50]
VALIDATION_GRID = Path('/content/asean_v21_validation_lambda_grid')
if VALIDATION_GRID.exists(): shutil.rmtree(VALIDATION_GRID)
VALIDATION_GRID.mkdir(parents=True, exist_ok=True)
lambda_rows = []
for lam in LAMBDA_GRID:
    run_dir = VALIDATION_GRID/f'lambda_{lam}'
    result = subprocess.run([sys.executable,str(REPO/'scripts/run_asean_v2_daily_backtest.py'),'--data-root',str(V2_DATA),'--prediction-file',str(V21_OUT/'validation_ensemble.npz'),'--run-dir',str(run_dir),'--risk-aversion',str(lam),'--cost-scenario','C0'], text=True, capture_output=True)
    print(f'lambda={lam} returncode={result.returncode}')
    if result.returncode != 0:
        print(result.stdout); print(result.stderr); raise RuntimeError('Validation lambda run failed.')
    perf = pd.read_csv(run_dir/'portfolio_metrics_summary.csv'); rel = pd.read_csv(run_dir/'reliability_metrics.csv')
    eligible = rel[(rel.covariance_fallback_rate.fillna(1) <= .10) & (rel.evaluation_coverage.fillna(0) >= .95)]
    lambda_rows.append({'risk_aversion':lam,'mean_country_sharpe':perf.loc[perf.country.isin(eligible.country),'annualized_net_sharpe'].mean(),'eligible_countries':len(eligible)})
lambda_table = pd.DataFrame(lambda_rows).sort_values(['eligible_countries','mean_country_sharpe'], ascending=[False,False])
display(lambda_table)
if lambda_table.empty or lambda_table.iloc[0].eligible_countries < 3: raise RuntimeError('No risk-aversion candidate passed reliability gates.')
SELECTED_LAMBDA = float(lambda_table.iloc[0].risk_aversion)
print('Validation-locked risk aversion:', SELECTED_LAMBDA)

In [ ]:
# Cell 8 — development portfolio backtests under C0/C1/C2
COST_RUNS = Path('/content/asean_v21_cost_scenarios')
if COST_RUNS.exists(): shutil.rmtree(COST_RUNS)
cost_rows = []
for scenario in ['C0','C1','C2']:
    run_dir = COST_RUNS/scenario
    command = [sys.executable,str(REPO/'scripts/run_asean_v2_daily_backtest.py'),'--data-root',str(V2_DATA),'--prediction-file',str(V21_OUT/'development_ensemble.npz'),'--run-dir',str(run_dir),'--risk-aversion',str(SELECTED_LAMBDA),'--cost-scenario',scenario]
    result = subprocess.run(command, text=True, capture_output=True)
    print(f'{scenario} returncode={result.returncode}')
    if result.stdout: print(result.stdout)
    if result.returncode != 0:
        if result.stderr: print(result.stderr)
        raise RuntimeError(f'{scenario} backtest failed.')
    perf = pd.read_csv(run_dir/'portfolio_metrics_summary.csv'); perf.insert(0,'cost_scenario',scenario); cost_rows.append(perf)
cost_summary = pd.concat(cost_rows, ignore_index=True)
display(cost_summary)

In [ ]:
# Cell 9 — save the complete continuation handoff to Drive
FINAL_OUT = DRIVE_OUT/'v2_1_portfolio'
if FINAL_OUT.exists(): shutil.rmtree(FINAL_OUT)
shutil.copytree(COST_RUNS, FINAL_OUT/'cost_scenarios')
cost_summary.to_csv(FINAL_OUT/'cost_sensitivity_summary.csv', index=False)
lambda_table.to_csv(FINAL_OUT/'validation_lambda_selection.csv', index=False)
shutil.copytree(ENSEMBLE_METRICS, FINAL_OUT/'ensemble_metrics', dirs_exist_ok=True)
print('Saved continuation handoff to:', FINAL_OUT)